In [ ]:
# ============================================
# 02_feature_engineering.ipynb
# STEP 5: FEATURE ENGINEERING (TF-IDF & Train-Test Split)
# ============================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import re

print("="*50)
print("STEP 5: FEATURE ENGINEERING")
print("="*50)

# ============================================
# Load processed data
# ============================================

print("\n📂 Loading data...")

# Try to load from saved file
try:
    df = pd.read_csv('data/sms_spam_processed.csv')
    print("✅ Data loaded from data/sms_spam_processed.csv")
except:
    # If file doesn't exist, load from URL and process
    print("⚠️ Processed file not found. Loading from URL...")
    url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
    df = pd.read_csv(url, sep='\t', header=None, names=['label', 'message'])

    # Clean text function
    def clean_text(text):
        text = text.lower()
        text = re.sub(r'[^\w\s]', '', text)
        text = re.sub(r'\d+', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    df['cleaned_message'] = df['message'].apply(clean_text)
    df['label_binary'] = df['label'].map({'spam': 1, 'ham': 0})
    print("✅ Data processed from URL")

print(f"\n✅ Data ready! Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

# ============================================
# Split features and target
# ============================================

print("\n" + "-"*40)
print("📊 SPLITTING DATA")
print("-"*40)

X = df['cleaned_message']
y = df['label_binary']

# Train-test split (80-20 with stratification)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n📈 Data Split Results:")
print(f"Training set: {len(X_train)} samples ({len(X_train)/len(X)*100:.1f}%)")
print(f"Testing set: {len(X_test)} samples ({len(X_test)/len(X)*100:.1f}%)")

print(f"\n📊 Training distribution:")
print(f"  Ham: {(y_train==0).sum()} messages")
print(f"  Spam: {(y_train==1).sum()} messages")

print(f"\n📊 Testing distribution:")
print(f"  Ham: {(y_test==0).sum()} messages")
print(f"  Spam: {(y_test==1).sum()} messages")

# ============================================
# TF-IDF Vectorization
# ============================================

print("\n" + "-"*40)
print("📝 TF-IDF VECTORIZATION")
print("-"*40)

print("\nApplying TF-IDF with:")
print("  - max_features: 5000")
print("  - ngram_range: (1, 2)")

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

# Fit on training data, transform both train and test
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f"\n✅ TF-IDF completed!")
print(f"Training features shape: {X_train_tfidf.shape}")
print(f"Testing features shape: {X_test_tfidf.shape}")

# Show some feature names (first 10)
feature_names = tfidf.get_feature_names_out()
print(f"\n📝 Sample features (first 10):")
print(feature_names[:10])

# ============================================
# Save features (for later use)
# ============================================

print("\n" + "-"*40)
print("💾 SAVING FEATURES")
print("-"*40)

# Save TF-IDF vectorizer
import joblib
import os

# Create directories if they don't exist
os.makedirs('models', exist_ok=True)
os.makedirs('data', exist_ok=True)

# Save vectorizer
joblib.dump(tfidf, 'models/tfidf_vectorizer.pkl')
print("✅ Saved: models/tfidf_vectorizer.pkl")

# Save sparse matrices
import scipy.sparse
scipy.sparse.save_npz('data/X_train_tfidf.npz', X_train_tfidf)
scipy.sparse.save_npz('data/X_test_tfidf.npz', X_test_tfidf)
print("✅ Saved: data/X_train_tfidf.npz")
print("✅ Saved: data/X_test_tfidf.npz")

# Save target variables
np.save('data/y_train.npy', y_train.values)
np.save('data/y_test.npy', y_test.values)
print("✅ Saved: data/y_train.npy")
print("✅ Saved: data/y_test.npy")

# Save the original train/test text for reference
pd.DataFrame({'message': X_train.values, 'label': y_train.values}).to_csv('data/train_data.csv', index=False)
pd.DataFrame({'message': X_test.values, 'label': y_test.values}).to_csv('data/test_data.csv', index=False)
print("✅ Saved: data/train_data.csv")
print("✅ Saved: data/test_data.csv")

# ============================================
# Summary
# ============================================

print("\n" + "="*50)
print("✅ FEATURE ENGINEERING COMPLETED!")
print("="*50)

print(f"""
📊 SUMMARY:
-----------
• Training samples: {X_train_tfidf.shape[0]}
• Testing samples: {X_test_tfidf.shape[0]}
• Features: {X_train_tfidf.shape[1]}
• Feature type: TF-IDF with n-grams (1,2)
• Vocabulary size: {len(tfidf.vocabulary_)}

📁 SAVED FILES:
-----------
• models/tfidf_vectorizer.pkl
• data/X_train_tfidf.npz
• data/X_test_tfidf.npz
• data/y_train.npy
• data/y_test.npy
• data/train_data.csv
• data/test_data.csv

🎯 NEXT STEP:
-----------
Run 03_model_training.ipynb to train models
""")

STEP 5: FEATURE ENGINEERING

📂 Loading data...
⚠️ Processed file not found. Loading from URL...
✅ Data processed from URL

✅ Data ready! Shape: (5572, 4)
Columns: ['label', 'message', 'cleaned_message', 'label_binary']

----------------------------------------
📊 SPLITTING DATA
----------------------------------------

📈 Data Split Results:
Training set: 4457 samples (80.0%)
Testing set: 1115 samples (20.0%)

📊 Training distribution:
  Ham: 3859 messages
  Spam: 598 messages

📊 Testing distribution:
  Ham: 966 messages
  Spam: 149 messages

----------------------------------------
📝 TF-IDF VECTORIZATION
----------------------------------------

Applying TF-IDF with:
  - max_features: 5000
  - ngram_range: (1, 2)

✅ TF-IDF completed!
Training features shape: (4457, 5000)
Testing features shape: (1115, 5000)

📝 Sample features (first 10):
['abiola' 'able' 'able to' 'about' 'about how' 'about me' 'about smiling'
 'about that' 'about the' 'about this']

-------------------------------------